In [ ]:
import pandas as pd
from xgboost import XGBRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler # اگرچه برای XGBoost کمتر حیاتی است، اما اضافه کردنش ضرری ندارد

In [ ]:
# 1. بارگذاری داده‌ها
train_df = pd.read_csv(r'train.csv')
test_df = pd.read_csv(r'test.csv')

print("داده‌های آموزشی بارگذاری شدند.")
print(f"شکل (Shape) مجموعه داده آموزش: {train_df.shape}")
print(f"شکل (Shape) مجموعه داده تست: {test_df.shape}\n")

In [ ]:
# 2. جدا کردن ویژگی‌ها (features) و هدف (target)
X = train_df.drop(columns=['electricity_consumption_kWh'])
y = train_df['electricity_consumption_kWh']

print("ویژگی‌ها (features) و هدف (target) از داده‌های آموزشی جدا شدند.")

In [ ]:
# 3. تقسیم داده‌های آموزش به بخش‌های آموزش و اعتبارسنجی
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42)

print("داده‌ها به بخش‌های آموزش و اعتبارسنجی تقسیم شدند.")

In [ ]:
# 4. ایجاد یک خط لوله (Pipeline) برای پیش‌پردازش و مدل‌سازی XGBoost
# اگر داده‌های شما خالی دارند، SimpleImputer مفید است.
# StandardScaler کمتر برای XGBoost ضروری است اما اضافه کردنش ضرری ندارد و کد را استانداردتر می‌کند.
pipeline_xgb = Pipeline([
    ('imputer', SimpleImputer(strategy='mean')), # یا 'median'
    ('scaler', StandardScaler()), # معمولا برای XGBoost لازم نیست ولی می توانید اضافه کنید
    ('xgbregressor', XGBRegressor(random_state=42))
])

In [ ]:
# 5. آموزش مدل با استفاده از Pipeline
print("آموزش مدل XGBoost با استفاده از Pipeline...")
pipeline_xgb.fit(X_train, y_train) # Pipeline خودش مقادیر خالی و مقیاس بندی را مدیریت می کند

In [ ]:
# 6. ارزیابی مدل روی مجموعه اعتبارسنجی (Validation Set)
y_val_pred_xgb = pipeline_xgb.predict(X_val)
r2_val_score_xgb = r2_score(y_val, y_val_pred_xgb)

print(f"R² Score روی مجموعه اعتبارسنجی با XGBoost (با Pipeline): {r2_val_score_xgb:.3f}")

In [ ]:
# 7. پیش‌بینی بر روی داده‌های تست
# Pipeline به طور خودکار داده های تست را هم پاکسازی و نرمال سازی می کند
y_test_pred_xgb = pipeline_xgb.predict(test_df)

In [ ]:
# 8. ایجاد و ذخیره فایل خروجی (submission)
submission_df = pd.DataFrame({
    'electricity_consumption_kWh': y_test_pred_xgb
})

submission_df.to_csv('submission_xgboost_pipeline.csv', index=False)

print("\nفایل submission_xgboost_pipeline.csv با موفقیت ساخته شد.")

In [ ]:
import joblib

# استخراج و ذخیره مستقیم مدل XGBoost از دل پایپ‌لاین
extracted_xgb = pipeline_xgb.named_steps['xgbregressor']
joblib.dump(extracted_xgb, 'xgboost_model.joblib')

print("✅ مدل XGBoost با موفقیت از پایپ‌لاین استخراج و با نام xgboost_model.joblib ذخیره شد!")